# EXPERIMENTO 5: EVALUACIÓN DEL RENDIMIENTO DE APACHE SPARK MEDIANTE DIFERENTES CANTIDADES DE PARTICIONES
**Responsable:** Lady Vanesa García Roldán
**Solución:** Big Data
**Plataforma:** Databricks
**Motor de procesamiento:** Apache Spark (PySpark)
## 1. Descripción del experimento

El presente experimento tiene como finalidad evaluar el rendimiento de Apache Spark mediante la ejecución de operaciones de procesamiento de datos utilizando diferentes cantidades de particiones.

Se trabajará con el conjunto de datos de citas médicas del proyecto, almacenado en la tabla `citas_original_100K`.

Se compararán cuatro escenarios experimentales con 1, 2, 4 y 8 particiones, manteniendo constantes el conjunto de datos y las operaciones ejecutadas.

## 2. Objetivo

Evaluar cómo influye la cantidad de particiones en el tiempo de procesamiento de Apache Spark, identificando la configuración que presenta el mejor rendimiento en el entorno de Databricks.

## 3. Hipótesis

La cantidad de particiones influye en el tiempo de procesamiento de Apache Spark, debido a que modifica la distribución de los datos entre las tareas de ejecución.

## 4. Escenarios experimentales

- Escenario A: 1 partición.
- Escenario B: 2 particiones.
- Escenario C: 4 particiones.
- Escenario D: 8 particiones.

## 5. Métricas de evaluación

- Tiempo de procesamiento (segundos).
- Tiempo promedio de ejecución.
- Tiempo mínimo y máximo.
- Cantidad de registros procesados correctamente.


## 1. Verificación del entorno y conexión al dataset

In [0]:

# Verificar el entorno de ejecución

print("EXPERIMENTO 5 - LADY GARCIA")
print("Motor de procesamiento:", spark.version)

# Verificar catálogo y esquema
spark.sql("""
    SELECT current_catalog(), current_schema()
""").show(truncate=False)

# Conectar con el dataset original
datos_originales = spark.table(
    "workspace.exp05_lady_garcia.citas_original_100K"
)

# Contar registros
total_registros = datos_originales.count()

print("Total de registros:", total_registros)

# Mostrar estructura de los datos
datos_originales.printSchema()

# Visualizar los primeros cinco registros
display(datos_originales.limit(5))


EXPERIMENTO 5 - LADY GARCIA
Motor de procesamiento: 4.2.0
+-----------------+----------------+
|current_catalog()|current_schema()|
+-----------------+----------------+
|workspace        |default         |
+-----------------+----------------+

Total de registros: 100000
root
 |-- id_cita: long (nullable = true)
 |-- fecha: date (nullable = true)
 |-- dni_paciente: long (nullable = true)
 |-- nombre_paciente: string (nullable = true)
 |-- cod_medico: string (nullable = true)
 |-- nombre_medico: string (nullable = true)
 |-- especialidad: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- duracion_min: long (nullable = true)
 |-- costo: long (nullable = true)



id_cita,fecha,dni_paciente,nombre_paciente,cod_medico,nombre_medico,especialidad,estado,duracion_min,costo
1,2026-08-21,72345678,Luis Castillo Mora,M003,Jorge Medina Rojas,Cardiología,Atendida,35,150
2,2026-08-19,71234567,María López Rivas,M005,Pedro Salazar Díaz,Traumatología,Atendida,40,140
3,2026-08-14,71234567,María López Rivas,M001,Carlos Ramírez Soto,Medicina General,Atendida,20,80
4,2026-08-02,78901234,Andrea Rojas Silva,M001,Carlos Ramírez Soto,Medicina General,Cancelada,15,80
5,2026-09-19,70234567,ELENA CAMPOS VERA,M002,Lucía Fernández Paredes,Pediatría,Atendida,40,100



## 2. Preparación y validación del dataset

In [0]:

from pyspark.sql import functions as F

# Seleccionar las columnas necesarias
datos = datos_originales.select(
    "id_cita",
    "fecha",
    "especialidad",
    "estado",
    "duracion_min",
    "costo"
)

# Verificar cantidad de registros
print("Total de registros:", datos.count())

# Verificar valores nulos
print("Valores nulos por columna:")

display(
    datos.select([
        F.count(
            F.when(F.col(c).isNull(), c)
        ).alias(c)
        for c in datos.columns
    ])
)

# Mostrar una muestra de los datos
display(datos.limit(5))


Total de registros: 100000
Valores nulos por columna:


id_cita,fecha,especialidad,estado,duracion_min,costo
0,0,0,0,0,0


id_cita,fecha,especialidad,estado,duracion_min,costo
1,2026-08-21,Cardiología,Atendida,35,150
2,2026-08-19,Traumatología,Atendida,40,140
3,2026-08-14,Medicina General,Atendida,20,80
4,2026-08-02,Medicina General,Cancelada,15,80
5,2026-09-19,Pediatría,Atendida,40,100


In [0]:

# Verificar identificadores duplicados

total_registros = datos.count()

registros_unicos = (
    datos.select("id_cita")
    .distinct()
    .count()
)

duplicados = total_registros - registros_unicos

print("Total de registros:", total_registros)
print("Identificadores únicos:", registros_unicos)
print("Registros duplicados:", duplicados)


Total de registros: 100000
Identificadores únicos: 99584
Registros duplicados: 416


In [0]:

# Identificar los ID de citas duplicados

duplicados_id = (
    datos
    .groupBy("id_cita")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

print("Cantidad de identificadores duplicados:")
print(duplicados_id.count())

# Mostrar una muestra de los identificadores repetidos
display(duplicados_id.limit(10))


Cantidad de identificadores duplicados:
416


id_cita,count
2387,2
1752,2
1999,2
2027,2
515,2
1681,2
7,2
1519,2
550,2
2735,2


In [0]:

# Analizar los registros con identificadores duplicados

ids_repetidos = duplicados_id.select("id_cita")

registros_repetidos = (
    datos
    .join(ids_repetidos, on="id_cita", how="inner")
    .orderBy("id_cita")
)

print("Registros asociados a identificadores repetidos:")

display(registros_repetidos.limit(20))


Registros asociados a identificadores repetidos:


id_cita,fecha,especialidad,estado,duracion_min,costo
7,2026-09-04,Ginecología,Cancelada,40,130
7,2026-09-04,Ginecología,Cancelada,40,130
515,2026-01-06,Cardiología,Programada,50,150
515,2026-01-06,Cardiología,Programada,50,150
550,2026-12-31,Pediatría,Atendida,30,100
550,2026-12-31,Pediatría,Atendida,30,100
1519,2025-07-25,Medicina General,Cancelada,15,80
1519,2025-07-25,Medicina General,Cancelada,15,80
1681,2025-08-30,Pediatría,Cancelada,35,100
1681,2025-08-30,Pediatría,Cancelada,35,100


## 3. Distribución de datos en diferentes particiones

In [0]:
# EXPERIMENTO 5 - DISTRIBUCIÓN DE PARTICIONES

from pyspark.sql import functions as F

# Definir los escenarios
escenarios = [1, 2, 4, 8]

print("DISTRIBUCIÓN DE DATOS POR ESCENARIO")
print("-----------------------------------")

for cantidad in escenarios:

    # Distribuir los datos
    datos_particionados = datos.repartition(cantidad)

    # Identificar las particiones mediante Spark SQL
    distribucion = (
        datos_particionados
        .select(
            F.spark_partition_id().alias("particion")
        )
        .groupBy("particion")
        .count()
        .orderBy("particion")
    )

    print(f"\nESCENARIO: {cantidad} PARTICIONES")

    display(distribucion)


DISTRIBUCIÓN DE DATOS POR ESCENARIO
-----------------------------------

ESCENARIO: 1 PARTICIONES


particion,count
0,100000



ESCENARIO: 2 PARTICIONES


particion,count
0,50000
1,50000



ESCENARIO: 4 PARTICIONES


particion,count
0,25000
1,25000
2,25000
3,25000



ESCENARIO: 8 PARTICIONES


particion,count
0,12500
1,12500
2,12500
3,12500
4,12500
5,12500
6,12500
7,12500


## 4. Ejecución y medición del rendimiento

In [0]:

import time
import statistics
from pyspark.sql import functions as F

# Función para ejecutar el procesamiento
def ejecutar_experimento(cantidad_particiones):

    inicio = time.perf_counter()

    resultado = (
        datos
        .repartition(cantidad_particiones)
        .groupBy("especialidad")
        .agg(
            F.count("*").alias("total_citas"),
            F.avg("duracion_min").alias("duracion_promedio"),
            F.sum("costo").alias("costo_total")
        )
        .collect()
    )

    fin = time.perf_counter()

    return fin - inicio, resultado

print("Función de medición preparada correctamente.")


Función de medición preparada correctamente.


In [0]:
# CALENTAMIENTO DE LOS ESCENARIOS

print("INICIANDO CALENTAMIENTO")
print("-----------------------")

for cantidad in [1, 2, 4, 8]:

    tiempo, resultado = ejecutar_experimento(cantidad)

    print(
        f"Particiones: {cantidad} | "
        f"Tiempo inicial: {tiempo:.4f} segundos"
    )

print("\nCalentamiento finalizado correctamente.")


INICIANDO CALENTAMIENTO
-----------------------
Particiones: 1 | Tiempo inicial: 0.8987 segundos
Particiones: 2 | Tiempo inicial: 0.5554 segundos
Particiones: 4 | Tiempo inicial: 0.6079 segundos
Particiones: 8 | Tiempo inicial: 0.5878 segundos

Calentamiento finalizado correctamente.


In [0]:

# EXPERIMENTO 5 - MEDICIONES OFICIALES

escenarios = [1, 2, 4, 8]
repeticiones = 5

tiempos = {
    cantidad: [] for cantidad in escenarios
}

print("INICIO DE LAS MEDICIONES OFICIALES")
print("----------------------------------")

for repeticion in range(repeticiones):

    print(f"\nREPETICIÓN {repeticion + 1}")

    for cantidad in escenarios:

        tiempo, resultado = ejecutar_experimento(cantidad)

        tiempos[cantidad].append(tiempo)

        total_procesado = sum(
            fila["total_citas"] for fila in resultado
        )

        print(
            f"Particiones: {cantidad} | "
            f"Tiempo: {tiempo:.4f} segundos | "
            f"Registros: {total_procesado}"
        )

print("\nMEDICIONES FINALIZADAS")


INICIO DE LAS MEDICIONES OFICIALES
----------------------------------

REPETICIÓN 1
Particiones: 1 | Tiempo: 0.5614 segundos | Registros: 100000
Particiones: 2 | Tiempo: 0.5739 segundos | Registros: 100000
Particiones: 4 | Tiempo: 0.6025 segundos | Registros: 100000
Particiones: 8 | Tiempo: 0.5654 segundos | Registros: 100000

REPETICIÓN 2
Particiones: 1 | Tiempo: 0.5053 segundos | Registros: 100000
Particiones: 2 | Tiempo: 0.5828 segundos | Registros: 100000
Particiones: 4 | Tiempo: 0.5558 segundos | Registros: 100000
Particiones: 8 | Tiempo: 0.5492 segundos | Registros: 100000

REPETICIÓN 3
Particiones: 1 | Tiempo: 0.4315 segundos | Registros: 100000
Particiones: 2 | Tiempo: 0.5227 segundos | Registros: 100000
Particiones: 4 | Tiempo: 0.4880 segundos | Registros: 100000
Particiones: 8 | Tiempo: 0.8937 segundos | Registros: 100000

REPETICIÓN 4
Particiones: 1 | Tiempo: 0.8709 segundos | Registros: 100000
Particiones: 2 | Tiempo: 0.5104 segundos | Registros: 100000
Particiones: 4 | Tie

## 5. Análisis estadístico de los tiempos de ejecución

In [0]:
import statistics
from pyspark.sql import functions as F

# Calcular estadísticas de cada escenario
resumen = []

for cantidad in [1, 2, 4, 8]:

    mediciones = tiempos[cantidad]

    resumen.append({
        "Particiones": cantidad,
        "Repeticiones": len(mediciones),
        "Tiempo_promedio": statistics.mean(mediciones),
        "Tiempo_minimo": min(mediciones),
        "Tiempo_maximo": max(mediciones),
        "Desviacion_estandar": statistics.stdev(mediciones)
    })

# Crear tabla de resultados
tabla_resultados = spark.createDataFrame(resumen)

display(tabla_resultados.orderBy("Particiones"))


Desviacion_estandar,Particiones,Repeticiones,Tiempo_maximo,Tiempo_minimo,Tiempo_promedio
0.17549278039325406,1,5,0.8709232949995567,0.43153972500022064,0.5686407494000377
0.032036877750900694,2,5,0.5827903719991809,0.510439783000038,0.5445198645998971
0.09590702978821697,4,5,0.7359230379997825,0.4880322389999492,0.5816133932001322
0.15515720997851326,8,5,0.8936804969998775,0.5316242139997485,0.6169650741998339


## 6. Comparación gráfica del rendimiento

In [0]:
# Mostrar la tabla de resultados experimentales

display(
    tabla_resultados.orderBy("Particiones")
)

Desviacion_estandar,Particiones,Repeticiones,Tiempo_maximo,Tiempo_minimo,Tiempo_promedio
0.17549278039325406,1,5,0.8709232949995567,0.43153972500022064,0.5686407494000377
0.032036877750900694,2,5,0.5827903719991809,0.510439783000038,0.5445198645998971
0.09590702978821697,4,5,0.7359230379997825,0.4880322389999492,0.5816133932001322
0.15515720997851326,8,5,0.8936804969998775,0.5316242139997485,0.6169650741998339


In [0]:

# Preparar datos para el gráfico comparativo

tabla_grafico = (
    tabla_resultados
    .withColumn(
        "Escenario",
        F.concat(
            F.col("Particiones").cast("string"),
            F.lit(" particiones")
        )
    )
    .select(
        "Escenario",
        "Tiempo_promedio"
    )
)

display(tabla_grafico)


Escenario,Tiempo_promedio
1 particiones,0.5686407494000377
2 particiones,0.5445198645998971
4 particiones,0.5816133932001322
8 particiones,0.6169650741998339


Databricks visualization. Run in Databricks to view.

## 7. Validación de consistencia de los resultados

In [0]:
# VALIDACIÓN DE CONSISTENCIA ENTRE ESCENARIOS

from math import isclose

print("VALIDACIÓN DE RESULTADOS")
print("------------------------")

# Utilizar 1 partición como escenario de referencia
tiempo_base, resultado_base = ejecutar_experimento(1)

# Ordenar los resultados por especialidad
base = sorted(resultado_base, key=lambda x: x["especialidad"])

for cantidad in [2, 4, 8]:

    tiempo, resultado = ejecutar_experimento(cantidad)

    actual = sorted(resultado, key=lambda x: x["especialidad"])

    # Comparar cantidad de grupos
    misma_cantidad = len(base) == len(actual)

    # Comparar resultados de cada especialidad
    equivalentes = misma_cantidad

    if equivalentes:
        for esperado, obtenido in zip(base, actual):

            if (
                esperado["especialidad"] != obtenido["especialidad"]
                or esperado["total_citas"] != obtenido["total_citas"]
                or esperado["costo_total"] != obtenido["costo_total"]
                or not isclose(
                    esperado["duracion_promedio"],
                    obtenido["duracion_promedio"],
                    rel_tol=1e-9,
                    abs_tol=1e-9
                )
            ):
                equivalentes = False
                break

    print(
        f"{cantidad} particiones: "
        f"{'RESULTADOS EQUIVALENTES' if equivalentes else 'DIFERENCIAS DETECTADAS'}"
    )

print("\nValidación finalizada.")


VALIDACIÓN DE RESULTADOS
------------------------
2 particiones: RESULTADOS EQUIVALENTES
4 particiones: RESULTADOS EQUIVALENTES
8 particiones: RESULTADOS EQUIVALENTES

Validación finalizada.
